# Polymarket Data Exploration
Exploring what data is available via the Polymarket public API — no account needed.

In [ ]:
import requests
import pandas as pd

GAMMA_BASE = "https://gamma-api.polymarket.com"
CLOB_BASE  = "https://clob.polymarket.com"

## 1. Browse active markets

In [ ]:
# Fetch top active markets sorted by volume
resp = requests.get(f"{GAMMA_BASE}/markets", params={
    "active": True,
    "closed": False,
    "limit": 20,
    "order": "volume24hr",
    "ascending": False,
})
markets = resp.json()
print(f"Fetched {len(markets)} markets")

In [ ]:
# Display key fields as a DataFrame
df = pd.DataFrame(markets)[
    ["id", "question", "volume", "volume24hr", "liquidity", "startDate", "endDate"]
]
df["volume"]     = pd.to_numeric(df["volume"], errors="coerce")
df["volume24hr"] = pd.to_numeric(df["volume24hr"], errors="coerce")
df["liquidity"]  = pd.to_numeric(df["liquidity"], errors="coerce")
df.sort_values("volume24hr", ascending=False)

## 2. Inspect a single market

In [ ]:
# Pick the highest-volume market
top = markets[0]
print("Question :", top["question"])
print("Volume   :", top["volume"])
print("Liquidity:", top["liquidity"])
print("End date :", top["endDate"])
print("\nOutcomes:")
for t in top.get("tokens", []):
    print(f"  {t['outcome']:10s}  token_id={t['token_id']}")

## 3. Order book (live bid/ask)

In [ ]:
# Grab the YES token id for the top market
yes_token = next(t for t in top["tokens"] if t["outcome"] == "Yes")
token_id  = yes_token["token_id"]

book = requests.get(f"{CLOB_BASE}/book", params={"token_id": token_id}).json()

bids = pd.DataFrame(book.get("bids", []), columns=["price", "size"]).astype(float)
asks = pd.DataFrame(book.get("asks", []), columns=["price", "size"]).astype(float)

print(f"Best bid: {bids['price'].max():.4f}  |  Best ask: {asks['price'].min():.4f}")
print(f"Spread  : {asks['price'].min() - bids['price'].max():.4f}")

## 4. Price history

In [ ]:
history = requests.get(f"{CLOB_BASE}/prices-history", params={
    "market":   token_id,
    "interval": "1h",  # options: 1m 5m 1h 6h 1d
    "fidelity": 60,
}).json()

ph = pd.DataFrame(history.get("history", []))
ph["t"] = pd.to_datetime(ph["t"], unit="s")
ph["p"] = pd.to_numeric(ph["p"])
ph = ph.rename(columns={"t": "timestamp", "p": "price"})
print(f"Price history rows: {len(ph)}")
ph.tail(10)

In [ ]:
ph.set_index("timestamp")["price"].plot(
    title=top["question"],
    ylabel="Implied probability (YES)",
    figsize=(12, 4),
)

## 5. Market categories & tags

In [ ]:
# What categories/tags exist?
all_tags = set()
for m in markets:
    for tag in m.get("tags", []):
        all_tags.add(tag.get("label", "") if isinstance(tag, dict) else tag)
print(sorted(all_tags))

## 6. Search markets by keyword

In [ ]:
keyword = "Trump"  # change to whatever topic interests you

resp = requests.get(f"{GAMMA_BASE}/markets", params={
    "active": True,
    "limit": 50,
    "q": keyword,
})
results = resp.json()
for m in results[:10]:
    vol = float(m.get("volume24hr") or 0)
    print(f"  [{vol:>10,.0f} vol/24h]  {m['question']}")